In [2]:
# !pip install ultralytics
# !pip install roboflow
# !pip install numpy==1.26.4

In [3]:
from ultralytics import YOLO

model = YOLO('yolov8l')
results = model.predict("/kaggle/input/football-clip/challenge-1140_1.mp4", save=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/67) /kaggle/input/football-clip/challenge-1140_1.mp4: 384x640 19 persons, 2 sports balls, 71.9ms
video 1/1 (frame 2/67) /kaggle/

In [4]:
print(len(results))
print(results[0])

67
ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plan

In [5]:
for box in results[0].boxes:
    print(box)
    break

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([0.], device='cuda:0')
conf: tensor([0.7276], device='cuda:0')
data: tensor([[1.4538e+03, 5.4774e+02, 1.4832e+03, 6.1342e+02, 7.2763e-01, 0.0000e+00]], device='cuda:0')
id: None
is_track: False
orig_shape: (1080, 1920)
shape: torch.Size([1, 6])
xywh: tensor([[1468.5051,  580.5824,   29.3269,   65.6763]], device='cuda:0')
xywhn: tensor([[0.7648, 0.5376, 0.0153, 0.0608]], device='cuda:0')
xyxy: tensor([[1453.8417,  547.7443, 1483.1686,  613.4205]], device='cuda:0')
xyxyn: tensor([[0.7572, 0.5072, 0.7725, 0.5680]], device='cuda:0')


## Football training - Get dataset

In [6]:
import roboflow
roboflow.__version__

'1.2.13'

In [8]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="ZeYwHkiSLBueVnaZ2Uje")
project = rf.workspace("minaehyeon").project("football-player-jrjtj")
version = project.version(5)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


In [9]:
with open("/kaggle/working/football-player-5/data.yaml") as f:
    print(f.read())

names:
- ball
- goalkeeper
- player
- referee
nc: 4
roboflow:
  license: CC BY 4.0
  project: football-player-jrjtj
  url: https://universe.roboflow.com/minaehyeon/football-player-jrjtj/dataset/5
  version: 5
  workspace: minaehyeon
test: ../test/images
train: ../train/images
val: ../valid/images



In [10]:
dataset.location

'/kaggle/working/football-player-5'

In [11]:
# solving dataset path confict
with open("/kaggle/working/football-player-5/data.yaml", "w") as f:
    f.write(
        "names:\n"
        "- ball\n"
        "- goalkeeper\n"
        "- player\n"
        "- referee\n\n"
        "nc: 4\n\n"
        "roboflow:\n"
          "license: CC BY 4.0\n"
          "project: football-player-jrjtj\n"
          "url: https://universe.roboflow.com/minaehyeon/football-player-jrjtj/dataset/5\n"
          "version: 5\n"
          "workspace: minaehyeon\n"
        "train: train/images\n"
        "val: valid/images\n"
        "test: test/images\n"
    )


In [12]:
with open("/kaggle/working/football-player-5/data.yaml") as f:
    print(f.read())

names:
- ball
- goalkeeper
- player
- referee

nc: 4

roboflow:
license: CC BY 4.0
project: football-player-jrjtj
url: https://universe.roboflow.com/minaehyeon/football-player-jrjtj/dataset/5
version: 5
workspace: minaehyeon
train: train/images
val: valid/images
test: test/images



In [13]:
# training yolo v8l model
!yolo task=detect mode=train model=yolov8l.pt data={dataset.location}/data.yaml epochs=20 imgsz=640

Ultralytics 8.4.14 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/football-player-5/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

In [14]:
print(f"{dataset.location}/data.yaml")

/kaggle/working/football-player-5/data.yaml
